## Ingestão BigQuery para Azure Blob Bronze

Notebook simplificado para execução local no Jupyter ou no Databricks. O fluxo carrega configurações, inicializa os clientes, exporta tabelas do BigQuery em Parquet e grava os arquivos no contêiner Bronze.


### 1. Dependências

Execute a célula abaixo apenas se o ambiente ainda não possuir as bibliotecas necessárias. Após instalar no Jupyter, reinicie o kernel.


In [ ]:
%pip install azure-storage-blob google-cloud-bigquery google-cloud-bigquery-storage db-dtypes pyarrow python-dotenv


In [ ]:
dbutils.library.restartPython()

### 2. Imports e detecção do ambiente


In [20]:
from pathlib import Path
from typing import Dict, List, Optional
import atexit
import datetime as dt
import os
import sys
import tempfile
import google.cloud.bigquery as bigquery

from azure.core.credentials import AzureNamedKeyCredential
from azure.storage.blob import BlobServiceClient


def detectar_databricks() -> bool:
    try:
        dbutils.widgets
        return True
    except (NameError, AttributeError):
        return False


IS_DATABRICKS = detectar_databricks()
print("✅ Ambiente:", "Databricks" if IS_DATABRICKS else "local")
print("✅ Diretório atual:", Path.cwd())


✅ Ambiente: local
✅ Diretório atual: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/notebooks


### 3. Localizar o projeto e carregar o `.env` local


In [21]:
def localizar_raiz_projeto() -> Path:
    atual = Path.cwd().resolve()

    for candidato in [atual, *atual.parents]:
        if (candidato / "src").is_dir():
            return candidato

    raise FileNotFoundError(
        "Pasta 'src' não encontrada no diretório atual ou nos diretórios pais."
    )


BASE_DIR = localizar_raiz_projeto()

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))


if not IS_DATABRICKS:
    from dotenv import load_dotenv

    env_path = BASE_DIR / ".env"

    if not env_path.is_file():
        raise FileNotFoundError(f"Arquivo .env não encontrado: {env_path}")

    if not load_dotenv(env_path, override=True):
        raise RuntimeError(f"Não foi possível carregar: {env_path}")

    print(f"✅ Ambiente local carregado: {env_path}")


from src.config.secrets import get_secret

print(f"✅ Raiz do projeto: {BASE_DIR}")


✅ Ambiente local carregado: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/.env
✅ Raiz do projeto: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28


### 4. Configurações e segredos


In [22]:
SECRET_SCOPE = "kvfiaptechprod"
DEFAULT_GCP_PROJECT_ID = "tough-medley-505300-k1"
DEFAULT_GCP_CREDENTIAL_FILE = "tough-medley-505300-k1-164371097431.json"

DEFAULT_TABLES = [
    "basedosdados.br_inep_avaliacao_alfabetizacao.uf",
    "basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil",
    "basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf",
    "basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio",
    "basedosdados.br_inep_avaliacao_alfabetizacao.municipio",
    "basedosdados.br_inep_avaliacao_alfabetizacao.alunos",
]


def exigir_valor(valor: Optional[str], nome: str) -> str:
    valor = str(valor).strip() if valor is not None else ""

    if not valor:
        raise RuntimeError(f"{nome} não configurado.")

    return valor


AZURE_STORAGE_ACCOUNT = exigir_valor(
    get_secret("AZURE-STORAGE-ACCOUNT", "AZURE_STORAGE_ACCOUNT"),
    "AZURE_STORAGE_ACCOUNT",
)

AZURE_STORAGE_KEY = exigir_valor(
    get_secret("AZURE-STORAGE-KEY", "AZURE_STORAGE_KEY"),
    "AZURE_STORAGE_KEY",
)

BRONZE_CONTAINER = exigir_valor(
    get_secret("AZURE-CONTAINER-BRONZE", "AZURE_CONTAINER_BRONZE"),
    "AZURE_CONTAINER_BRONZE",
)

GOOGLE_CREDENTIALS_JSON = get_secret(
    "GOOGLE-APPLICATION-CREDENTIALS-JSON",
    "GOOGLE_APPLICATION_CREDENTIALS_JSON",
)

GCP_PROJECT_ID = os.getenv(
    "GCP_PROJECT_ID",
    DEFAULT_GCP_PROJECT_ID,
).strip()

print(f"✅ Storage Account: {AZURE_STORAGE_ACCOUNT}")
print(f"✅ Container Bronze: {BRONZE_CONTAINER}")
print(f"✅ Projeto GCP: {GCP_PROJECT_ID}")
print(f"✅ Storage Key presente: {bool(AZURE_STORAGE_KEY)}")


✅ Storage Account: stfiaptechchallenge2iast
✅ Container Bronze: bronze
✅ Projeto GCP: tough-medley-505300-k1
✅ Storage Key presente: True


### 5. Credencial Google e tabelas


In [23]:
TEMP_CREDENTIAL_PATH = None


def remover_credencial_temporaria() -> None:
    if TEMP_CREDENTIAL_PATH:
        caminho = Path(TEMP_CREDENTIAL_PATH)
        if caminho.is_file():
            caminho.unlink()


atexit.register(remover_credencial_temporaria)


def configurar_credencial_google() -> Path:
    global TEMP_CREDENTIAL_PATH

    if GOOGLE_CREDENTIALS_JSON:
        with tempfile.NamedTemporaryFile(
            mode="w",
            suffix=".json",
            delete=False,
            encoding="utf-8",
        ) as arquivo:
            arquivo.write(GOOGLE_CREDENTIALS_JSON)
            TEMP_CREDENTIAL_PATH = arquivo.name

        caminho = Path(TEMP_CREDENTIAL_PATH)
        origem = "Key Vault"
    else:
        caminho = BASE_DIR / "credenciais" / DEFAULT_GCP_CREDENTIAL_FILE
        origem = "arquivo local"

    if not caminho.is_file():
        raise FileNotFoundError(f"Credencial Google não encontrada: {caminho}")

    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(caminho)
    print(f"✅ Credencial Google carregada de: {origem}")
    return caminho


def carregar_tabelas() -> List[str]:
    if not IS_DATABRICKS:
        return DEFAULT_TABLES.copy()

    dbutils.widgets.text("TABLES", "")
    valor = dbutils.widgets.get("TABLES").strip()

    if not valor:
        return DEFAULT_TABLES.copy()

    tabelas = [item.strip() for item in valor.split(",") if item.strip()]
    return tabelas or DEFAULT_TABLES.copy()


GCP_CREDENTIAL_PATH = configurar_credencial_google()
TABLES = carregar_tabelas()

print(f"✅ Tabelas configuradas: {len(TABLES)}")
for tabela in TABLES:
    print(f"   • {tabela}")


✅ Credencial Google carregada de: arquivo local
✅ Tabelas configuradas: 6
   • basedosdados.br_inep_avaliacao_alfabetizacao.uf
   • basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil
   • basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf
   • basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio
   • basedosdados.br_inep_avaliacao_alfabetizacao.municipio
   • basedosdados.br_inep_avaliacao_alfabetizacao.alunos


### 6. Inicializar e validar os clientes


In [24]:
def criar_cliente_bigquery():
    cliente = bigquery.Client.from_service_account_json(
        str(GCP_CREDENTIAL_PATH),
        project=GCP_PROJECT_ID,
    )
    print("✅ Cliente BigQuery inicializado.")
    return cliente


def criar_cliente_blob():
    credencial = AzureNamedKeyCredential(
        name=AZURE_STORAGE_ACCOUNT,
        key=AZURE_STORAGE_KEY,
    )

    cliente = BlobServiceClient(
        account_url=(
            f"https://{AZURE_STORAGE_ACCOUNT}.blob.core.windows.net"
        ),
        credential=credencial,
    )

    cliente.get_account_information()

    container = cliente.get_container_client(BRONZE_CONTAINER)
    container.get_container_properties()

    print("✅ Cliente Azure Blob autenticado.")
    print(f"✅ Container acessível: {BRONZE_CONTAINER}")
    return cliente


bq_client = criar_cliente_bigquery()
blob_service_client = criar_cliente_blob()


✅ Cliente BigQuery inicializado.
✅ Cliente Azure Blob autenticado.
✅ Container acessível: bronze


### 7. Função de exportação


In [25]:
def exportar_tabela(
    source_table: str,
    blob_name: str,
) -> Dict:
    if not source_table:
        raise ValueError(
            "Tabela de origem não informada."
        )

    if not blob_name:
        raise ValueError(
            "Nome do blob não informado."
        )

    query = (
        f"SELECT * "
        f"FROM `{source_table}`"
    )

    print(
        f"▶ Consultando: {source_table}"
    )

    dataframe = (
        bq_client
        .query(
            query,
            location="US",
        )
        .to_dataframe()
    )

    if dataframe.empty:
        raise ValueError(
            f"Nenhum registro retornado por "
            f"{source_table}."
        )

    # =========================
    # Colunas de auditoria
    # =========================

    dataframe["_ingested_at"] = (
        dt.datetime.now(
            dt.timezone.utc
        ).isoformat()
    )

    dataframe["_source_table"] = (
        source_table
    )

    # =========================
    # Pasta Bronze local
    # =========================

    local_bronze_path = (BASE_DIR/ "dados" / "bronze")
    local_bronze_path.mkdir(parents=True,exist_ok=True,)
    parquet_file = (local_bronze_path/ blob_name)

    # =========================
    # Gerar Parquet local
    # =========================

    dataframe.to_parquet(
        parquet_file,
        index=False,
    )

    if not parquet_file.is_file():
        raise FileNotFoundError(
            f"Parquet não criado: "
            f"{parquet_file}"
        )

    tamanho_local = (
        parquet_file.stat().st_size
    )

    print(
        f"✅ Parquet local: "
        f"{parquet_file}"
    )

    # =========================
    # Upload para Azure Bronze
    # =========================

    blob_client = (
        blob_service_client
        .get_blob_client(
            container=BRONZE_CONTAINER,
            blob=blob_name,
        )
    )

    with parquet_file.open("rb") as arquivo:
        blob_client.upload_blob(
            data=arquivo,
            overwrite=True,
        )

    propriedades = (
        blob_client.get_blob_properties()
    )

    if propriedades.size != tamanho_local:
        raise RuntimeError(
            "Tamanho divergente. "
            f"Local: {tamanho_local} bytes. "
            f"Azure: {propriedades.size} bytes."
        )

    resultado = {
        "success": True,
        "table": source_table,
        "container": BRONZE_CONTAINER,
        "blob": blob_name,
        "local_file": str(parquet_file),
        "records": len(dataframe),
        "columns": len(dataframe.columns),
        "size_bytes": propriedades.size,
    }

    print(
        f"✅ Blob enviado: {blob_name}"
    )

    print(
        f"✅ Registros: "
        f"{len(dataframe):,}"
    )

    print(
        f"✅ Colunas: "
        f"{len(dataframe.columns)}"
    )

    print(
        f"✅ Tamanho: "
        f"{propriedades.size / 1024:.2f} KB"
    )

    return resultado

### 8. Executar ingestão batch


In [26]:
def executar_ingestao(tabelas: List[str]) -> List[Dict]:
    data_execucao = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%d")
    resultados = []
    falhas = []

    print("========== INGESTÃO ==========")
    print(f"✅ Data UTC: {data_execucao}")
    print(f"✅ Total de tabelas: {len(tabelas)}")

    for indice, tabela in enumerate(tabelas, start=1):
        sufixo = tabela.rsplit(".", maxsplit=1)[-1]
        blob_name = f"{data_execucao}_{sufixo}.parquet"

        print()
        print(f"[{indice}/{len(tabelas)}] {tabela}")

        try:
            resultados.append(
                exportar_tabela(
                    source_table=tabela,
                    blob_name=blob_name,
                )
            )
        except Exception as exc:
            erro = f"{type(exc).__name__}: {exc}"
            falhas.append({"table": tabela, "error": erro})
            print(f"❌ Falha: {erro}")

    print()
    print("========== RESUMO ==========")
    print(f"✅ Sucessos: {len(resultados)}")
    print(f"❌ Falhas: {len(falhas)}")
    print(f"✅ Container: {BRONZE_CONTAINER}")

    if falhas:
        for falha in falhas:
            print(f"- {falha['table']}: {falha['error']}")

        raise RuntimeError(
            f"Processo concluído com {len(falhas)} falha(s)."
        )

    print("✅ Todas as tabelas foram exportadas com sucesso.")
    return resultados


resultados = executar_ingestao(TABLES)


========== INGESTÃO ==========
✅ Data UTC: 2026-10-18
✅ Total de tabelas: 6

[1/6] basedosdados.br_inep_avaliacao_alfabetizacao.uf
▶ Consultando: basedosdados.br_inep_avaliacao_alfabetizacao.uf
✅ Parquet local: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze/2026-10-18_uf.parquet
✅ Blob enviado: 2026-10-18_uf.parquet
✅ Registros: 145
✅ Colunas: 17
✅ Tamanho: 17.46 KB

[2/6] basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil
▶ Consultando: basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil
✅ Parquet local: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze/2026-10-18_meta_alfabetizacao_brasil.parquet
✅ Blob enviado: 2026-10-18_meta_alfabetizacao_brasil.parquet
✅ Registros: 3
✅ Colunas: 13
✅ Tamanho: 9.22 KB

[3/6] basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf
▶ Consultando: basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf
✅ Parquet local: /home/linux/projetos

### 9. Visualizar o resumo


In [27]:
import pandas as pd

resumo = pd.DataFrame(resultados)
resumo


,success,table,container,blob,local_file,records,columns,size_bytes
0,True,basedosdados.br_inep_avaliacao_alfabetizacao.uf,bronze,2026-10-18_uf.parquet,/home/linux/projetos/postech-2iast-fase3-model...,145,17,17875
1,True,basedosdados.br_inep_avaliacao_alfabetizacao.m...,bronze,2026-10-18_meta_alfabetizacao_brasil.parquet,/home/linux/projetos/postech-2iast-fase3-model...,3,13,9438
2,True,basedosdados.br_inep_avaliacao_alfabetizacao.m...,bronze,2026-10-18_meta_alfabetizacao_uf.parquet,/home/linux/projetos/postech-2iast-fase3-model...,81,14,12401
3,True,basedosdados.br_inep_avaliacao_alfabetizacao.m...,bronze,2026-10-18_meta_alfabetizacao_municipio.parquet,/home/linux/projetos/postech-2iast-fase3-model...,10704,15,222829
4,True,basedosdados.br_inep_avaliacao_alfabetizacao.m...,bronze,2026-10-18_municipio.parquet,/home/linux/projetos/postech-2iast-fase3-model...,23995,17,497738
5,True,basedosdados.br_inep_avaliacao_alfabetizacao.a...,bronze,2026-10-18_alunos.parquet,/home/linux/projetos/postech-2iast-fase3-model...,3867999,14,70913390
